# DPAmplify — Interactive Demo

This notebook demonstrates the core ideas behind the DPAmplify attack:
how a Byzantine client exploits the zero-bias property of the Gaussian
DP mechanism to steer a federated model toward an arbitrary target
direction, while remaining statistically indistinguishable from honest
clients. We also compare the closed-form SNR upper bound (Theorem 1a)
against the tighter empirical bound (Theorem 1b), and show how
passive parameter estimation enables the attack even when C and σ are
unknown.

## 1. The Gaussian DP Mechanism

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from theory.dp_mechanism import DPMechanism

mech = DPMechanism(C=1.0, sigma=0.1)

# Test clipping
g_above = np.array([2.0, 0.0, 0.0])   # norm = 2  > C=1
g_below = np.array([0.5, 0.0, 0.0])   # norm = 0.5 < C=1

g_above_clipped = mech.clip(g_above)
g_below_clipped = mech.clip(g_below)

print(f'g_above           norm = {np.linalg.norm(g_above):.3f}')
print(f'clip(g_above, 1)  norm = {np.linalg.norm(g_above_clipped):.3f}  (clipped to C=1)')
print()
print(f'g_below           norm = {np.linalg.norm(g_below):.3f}')
print(f'clip(g_below, 1)  norm = {np.linalg.norm(g_below_clipped):.3f}  (unchanged)')

## 2. The Adversarial Gradient

In [ ]:
from attack.gradient_optimizer import GradientOptimizer

# 100-dimensional target: 5 * e_1
D = 100
g_target = np.zeros(D)
g_target[0] = 5.0

opt   = GradientOptimizer(g_target, C=1.0)
g_adv = opt.compute_g_adv()

# Verify zero-bias via Monte Carlo
rng = np.random.default_rng(42)
samples = mech.sample_outputs(g_adv, 10_000, rng)
mean_output = samples.mean(axis=0)

cosine_sim = float(np.dot(mean_output, g_adv) / (
    np.linalg.norm(mean_output) * np.linalg.norm(g_adv) + 1e-12
))

print(f'||g_adv||₂            = {np.linalg.norm(g_adv):.6f}  (should equal C=1.0)')
print(f'E[M_DP(g_adv)][0]     = {mean_output[0]:.6f}  (should ≈ g_adv[0]={g_adv[0]:.4f})')
print(f'Cosine similarity     = {cosine_sim:.6f}  (should ≈ 1.0)')

## 3. SNR — Upper Bound vs Tight Bound

In [ ]:
from theory.snr_analysis import (
    compute_attack_snr_upper_bound,
    compute_attack_snr_tight,
)

n         = 20
sigma     = 0.1
var_honest = 0.01   # empirically estimated in poc.py (var_honest ≈ 0.010)
k_values  = list(range(1, 10))

snr_upper = [compute_attack_snr_upper_bound(k, n, 1.0, sigma) for k in k_values]
snr_tight = [compute_attack_snr_tight(k, n, 1.0, sigma, var_honest) for k in k_values]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_values, snr_upper, 's--', color='tab:red',  label='Upper bound (Thm 1a)')
ax.plot(k_values, snr_tight, '^-',  color='tab:blue', label='Tight bound (Thm 1b)')
ax.axhline(1.0, color='grey', linestyle=':', label='SNR = 1 (attack threshold)')

# Mark the SNR=1 crossing for each curve
for snr_vals, color, label in [
    (snr_upper, 'tab:red',  'Upper'),
    (snr_tight, 'tab:blue', 'Tight'),
]:
    for i, (k, s) in enumerate(zip(k_values, snr_vals)):
        if s >= 1.0:
            ax.annotate(f'{label} ≥ 1\nat k={k}',
                        xy=(k, s), xytext=(k + 0.3, s + 0.5),
                        fontsize=8, color=color,
                        arrowprops=dict(arrowstyle='->', color=color))
            break

ax.set_xlabel('k (Byzantine clients)')
ax.set_ylabel('SNR')
ax.set_title(f'Attack SNR vs k  (n={n}, σ={sigma}, Var_h={var_honest})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Passive Parameter Estimation

In [ ]:
from attack.parameter_estimator import PassiveParameterEstimator

true_C     = 1.0
true_sigma = 0.1
rng2 = np.random.default_rng(7)

est = PassiveParameterEstimator(history_window=10)
observations = []
C_hats, sigma_hats = [], []

for i in range(25):
    norm = max(rng2.normal(true_C, true_sigma), 0.0)
    est.update(norm)
    observations.append(norm)
    if est.is_ready():
        C_hats.append(est.estimate_C())
        sigma_hats.append(est.estimate_sigma())
    else:
        C_hats.append(None)
        sigma_hats.append(None)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
rounds = list(range(1, 26))

ax = axes[0]
ax.scatter(rounds, observations, s=20, color='grey', alpha=0.6, label='Observed norms')
C_valid = [(r, v) for r, v in zip(rounds, C_hats) if v is not None]
ax.plot([r for r,v in C_valid], [v for r,v in C_valid], 'b-o', ms=4, label='Ĉ (running estimate)')
ax.axhline(true_C, color='black', linestyle='--', label=f'True C={true_C}')
ax.set_xlabel('Round'); ax.set_ylabel('Value'); ax.set_title('C estimation')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[1]
s_valid = [(r, v) for r, v in zip(rounds, sigma_hats) if v is not None]
ax.plot([r for r,v in s_valid], [v for r,v in s_valid], 'r-o', ms=4, label='σ̂ (running estimate)')
ax.axhline(true_sigma, color='black', linestyle='--', label=f'True σ={true_sigma}')
ax.set_xlabel('Round'); ax.set_ylabel('Value'); ax.set_title('σ estimation')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('Passive Parameter Estimation', y=1.02)
plt.tight_layout()
plt.show()

final = est.get_estimates()
print(f"Final estimate:  Ĉ={final['C']:.4f} (true {true_C}), "
      f"σ̂={final['sigma']:.4f} (true {true_sigma})")

## 5. Toy Simulation (2D, n=5, k=1)

In [ ]:
# 2-D federation: 5 clients, 1 Byzantine, 30 rounds
# Attack target: direction [1, 0]
# Plots trajectory of global model (initialised at origin)
# vs clean baseline (no Byzantine client)

N_ROUNDS = 30
N_TOT    = 5
K        = 1
C        = 1.0
SIGMA    = 0.1
LR       = 0.1   # server learning rate

mech2d = DPMechanism(C=C, sigma=SIGMA)
rng3   = np.random.default_rng(0)

g_target_2d = np.array([1.0, 0.0])
opt2d = GradientOptimizer(g_target_2d, C=C)
g_adv_2d = opt2d.compute_g_adv()

theta_attack = np.array([0.0, 0.0])
theta_clean  = np.array([0.0, 0.0])
traj_attack  = [theta_attack.copy()]
traj_clean   = [theta_clean.copy()]

for _ in range(N_ROUNDS):
    # Attack scenario
    honest_updates  = [mech2d.apply(rng3.normal(0, 0.3, 2), rng3)
                       for _ in range(N_TOT - K)]
    byz_updates     = [mech2d.apply(g_adv_2d, rng3) for _ in range(K)]
    agg_attack      = np.mean(honest_updates + byz_updates, axis=0)
    theta_attack    = theta_attack + LR * agg_attack
    traj_attack.append(theta_attack.copy())

    # Clean baseline
    clean_updates   = [mech2d.apply(rng3.normal(0, 0.3, 2), rng3)
                       for _ in range(N_TOT)]
    agg_clean       = np.mean(clean_updates, axis=0)
    theta_clean     = theta_clean + LR * agg_clean
    traj_clean.append(theta_clean.copy())

traj_attack = np.array(traj_attack)
traj_clean  = np.array(traj_clean)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(traj_attack[:, 0], traj_attack[:, 1], 'r-o', ms=4, label='With DPAmplify (k=1)')
ax.plot(traj_clean[:, 0],  traj_clean[:, 1],  'b-s', ms=4, label='Clean baseline')
ax.annotate('', xy=(1.5, 0), xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='darkred', lw=2))
ax.text(1.5, 0.05, 'Target →', color='darkred', fontsize=9)
ax.scatter([0], [0], color='black', s=60, zorder=5, label='Start')
ax.set_xlabel('θ₁');  ax.set_ylabel('θ₂')
ax.set_title(f'Global model trajectory (2D, {N_ROUNDS} rounds, k={K}/{N_TOT})')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_aspect('equal')
plt.tight_layout()
plt.show()

final_proj_attack = float(traj_attack[-1] @ g_target_2d)
final_proj_clean  = float(traj_clean[-1]  @ g_target_2d)
print(f'Final projection onto target:  attack={final_proj_attack:.4f},  clean={final_proj_clean:.4f}')